# Logistic Regression for Grain Disease Recognition

## Data Preprocessing

In [1]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
)
from tqdm import tqdm
import joblib

In [2]:
GRAIN_TYPES = ["maize", "rice"]
MAIZE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_HD", "7_IM"]
RICE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_UN", "7_IM"]

We need to turn the pixels of the image into a feature vector to feed into the training model. We decided to use the HSV histogram as the embedding function to create the feature vector.

In [3]:
from logistic_regression_lib import collect_embeddings

## Train Logistic Regression Model

Collect embeddings from training images and train a logistic regression classifier. For this, we combine the grain type and category in order to try and train a generalized model for both rice and maize.

We use the HSV values for extracting the features of the image to get the color information of the image. Due to this, there is no spatial information in the embedding of the image.

We will also use both the train and validation sets for this as we will use K-fold cross validation and it will automatically segment the dataset into train and val sets.

In [4]:
# Collect training and test data
X_train_maize, y_train_maize = collect_embeddings("maize", "train")
X_train_rice, y_train_rice = collect_embeddings("rice", "train")
X_val_maize, y_val_maize = collect_embeddings("maize", "val")
X_val_rice, y_val_rice = collect_embeddings("rice", "val")

X_train = np.concatenate((X_train_maize, X_train_rice, X_val_maize, X_val_rice))
y_train = np.concatenate((y_train_maize, y_train_rice, y_val_maize, y_val_rice))
print(f"\nTraining data shape: {X_train.shape}")
print(f"Number of classes: {len(set(y_train))}")
print(f"Classes: {sorted(set(y_train))}")

Processing train images: 100%|██████████| 12240/12240 [01:49<00:00, 111.29it/s]


Processing train images: 100%|██████████| 22155/22155 [00:45<00:00, 487.57it/s]


Processing val images: 100%|██████████| 2160/2160 [00:19<00:00, 110.22it/s]


Processing val images: 100%|██████████| 3907/3907 [00:08<00:00, 479.03it/s]



Training data shape: (40462, 512)
Number of classes: 14
Classes: [np.str_('maize_0_NOR'), np.str_('maize_1_F&S'), np.str_('maize_2_SD'), np.str_('maize_3_MY'), np.str_('maize_4_AP'), np.str_('maize_5_BN'), np.str_('maize_6_HD'), np.str_('rice_0_NOR'), np.str_('rice_1_F&S'), np.str_('rice_2_SD'), np.str_('rice_3_MY'), np.str_('rice_4_AP'), np.str_('rice_5_BN'), np.str_('rice_6_UN')]


In [5]:
# Collect test data
X_test_maize, y_test_maize = collect_embeddings("maize", "test")
X_test_rice, y_test_rice = collect_embeddings("rice", "test")
X_test = np.concatenate((X_test_maize, X_test_rice))
y_test = np.concatenate((y_test_maize, y_test_rice))

print(f"Test data shape: {X_test.shape}")
print(f"Number of test classes: {len(set(y_test))}")
print(f"Test classes: {sorted(set(y_test))}")

Processing test images: 100%|██████████| 1600/1600 [00:14<00:00, 109.79it/s]


Processing test images: 100%|██████████| 2900/2900 [00:06<00:00, 464.23it/s]


Test data shape: (4500, 512)
Number of test classes: 14
Test classes: [np.str_('maize_0_NOR'), np.str_('maize_1_F&S'), np.str_('maize_2_SD'), np.str_('maize_3_MY'), np.str_('maize_4_AP'), np.str_('maize_5_BN'), np.str_('maize_6_HD'), np.str_('rice_0_NOR'), np.str_('rice_1_F&S'), np.str_('rice_2_SD'), np.str_('rice_3_MY'), np.str_('rice_4_AP'), np.str_('rice_5_BN'), np.str_('rice_6_UN')]


And lets encode the labels to integers for the logistic regression to work

In [6]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

print(f"Label encoding: {dict(enumerate(label_encoder.classes_))}")

Label encoding: {0: np.str_('maize_0_NOR'), 1: np.str_('maize_1_F&S'), 2: np.str_('maize_2_SD'), 3: np.str_('maize_3_MY'), 4: np.str_('maize_4_AP'), 5: np.str_('maize_5_BN'), 6: np.str_('maize_6_HD'), 7: np.str_('rice_0_NOR'), 8: np.str_('rice_1_F&S'), 9: np.str_('rice_2_SD'), 10: np.str_('rice_3_MY'), 11: np.str_('rice_4_AP'), 12: np.str_('rice_5_BN'), 13: np.str_('rice_6_UN')}


In [7]:
# Encode test labels
y_test_encoded = label_encoder.transform(y_test)
print(f"\nTest set size: {X_test.shape[0]} samples")


Test set size: 4500 samples


## Training the Model with Hyperparameter Tuning using Random Search

Now let's optimize the logistic regression model by finding the best hyperparameters using GridSearchCV. We'll tune:
- **C**: Regularization strength (lower = stronger regularization)
- **penalty**: Type of regularization (L1 or L2)
- **class_weight**: Whether to balance class weights
- **solver**: Algorithm to use for optimizatio

We will use the weighted F1-score for scoring the hyperparameters in the random search because of the imbalanced classes in the dataset.

In [8]:
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)

param_dist = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l2"],
    "classifier__class_weight": [None, "balanced"],
    "classifier__solver": ["lbfgs"],
}

n_iter = 20  # Number of parameter combinations to sample

print("Starting Random Search with Pipeline...")
print(f"Testing {n_iter} random combinations from possible parameter space")

# Random search with cross-validation using F1-weighted scoring
random_search = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=n_iter,  # Number of combinations to try
    cv=5,  # 5-fold cross-validation
    scoring="f1_weighted",  # Use F1 instead of accuracy for imbalanced data
    n_jobs=-1,  # Use all CPU cores
    verbose=2,
    random_state=42,
)

random_search.fit(X_train, y_train_encoded)

print("\nRANDOM SEARCH RESULTS")
print(f"\nBest hyperparameters: {random_search.best_params_}")
print(f"Best cross-validation F1-score: {random_search.best_score_:.4f}")

Starting Random Search with Pipeline...
Testing 20 random combinations from possible parameter space
Fitting 5 folds for each of 12 candidates, totalling 60 fits


/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depreca

[CV] END classifier__C=0.001, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 1.0min
[CV] END classifier__C=0.001, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 1.0min
[CV] END classifier__C=0.001, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 1.0min


/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depreca

...

[CV] END classifier__C=10, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=lbfgs; total time=10.7min


/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END classifier__C=100, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 8.3min
[CV] END classifier__C=100, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 8.8min
[CV] END classifier__C=100, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 9.1min
[CV] END classifier__C=10, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=lbfgs; total time=10.7min
[CV] END classifier__C=100, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 9.0min
[CV] END classifier__C=100, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=lbfgs; total time= 9.8min
[CV] END classifier__C=10, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=lbfgs; total time=10.2min
[CV] END classifier__C=100, classifier__class_weight=balanced, classifier__penalty=l2, classifier_

/home/xandreiathome/.conda/envs/python3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



RANDOM SEARCH RESULTS

Best hyperparameters: {'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__class_weight': None, 'classifier__C': 1}
Best cross-validation F1-score: 0.8879


In [ ]:
# Get the best model (pipeline)
best_pipeline = random_search.best_estimator_

# Evaluate on test set (pipeline handles scaling)
y_test_pred_best = best_pipeline.predict(X_test)

best_accuracy = best_pipeline.score(X_test, y_test_encoded)
best_f1 = f1_score(y_test_encoded, y_test_pred_best, average="weighted")
best_recall = recall_score(y_test_encoded, y_test_pred_best, average="weighted")
best_precision = precision_score(y_test_encoded, y_test_pred_best, average="weighted")

print("\nTEST SET PERFORMANCE - BEST MODEL")
print(f"Test Accuracy (weighted):  {best_accuracy:.4f}")
print(f"Test F1-Score (weighted):  {best_f1:.4f}")
print(f"Test Recall (weighted):    {best_recall:.4f}")
print(f"Test Precision (weighted): {best_precision:.4f}")


TEST SET PERFORMANCE - BEST MODEL
Test Accuracy:  0.8704
Test F1-Score:  0.8685
Test Recall:    0.8704
Test Precision: 0.8710


In [10]:
# Visualize top results
results_df = pd.DataFrame(random_search.cv_results_)
results_df = results_df[
    [
        "param_classifier__C",
        "param_classifier__penalty",
        "param_classifier__class_weight",
        "param_classifier__solver",
        "mean_test_score",
        "std_test_score",
    ]
]
results_df = results_df.sort_values("mean_test_score", ascending=False)

print("\nTOP 10 PARAMETER COMBINATIONS (by mean F1-score)")
print(results_df.head(10).to_string(index=False))


TOP 10 PARAMETER COMBINATIONS (by mean F1-score)
 param_classifier__C param_classifier__penalty param_classifier__class_weight param_classifier__solver  mean_test_score  std_test_score
                1.00                        l2                            NaN                    lbfgs         0.887937        0.001339
               10.00                        l2                            NaN                    lbfgs         0.886526        0.002037
              100.00                        l2                            NaN                    lbfgs         0.885071        0.002427
                0.10                        l2                            NaN                    lbfgs         0.884935        0.001519
                0.01                        l2                            NaN                    lbfgs         0.868379        0.002332
                1.00                        l2                       balanced                    lbfgs         0.861263        0.00287

In [11]:
# Display per-class metrics for best model
from sklearn.metrics import classification_report


print("\nPER-CLASS METRICS - BEST MODEL")
best_report = classification_report(
    y_test_encoded, y_test_pred_best, target_names=label_encoder.classes_, digits=4
)
print(best_report)

# Save the best model (pipeline)
joblib.dump(best_pipeline, "models/logistic_regression_tuned.pkl")
print("\nBest tuned pipeline saved to models/logistic_regression_tuned.pkl")


PER-CLASS METRICS - BEST MODEL
              precision    recall  f1-score   support

 maize_0_NOR     0.8987    0.9670    0.9316      1000
 maize_1_F&S     0.8878    0.8700    0.8788       100
  maize_2_SD     0.9286    0.7800    0.8478       100
  maize_3_MY     0.6786    0.3800    0.4872       100
  maize_4_AP     0.5833    0.5600    0.5714       100
  maize_5_BN     0.7692    0.8000    0.7843       100
  maize_6_HD     0.9348    0.8600    0.8958       100
  rice_0_NOR     0.9630    0.9490    0.9559      2000
  rice_1_F&S     0.5319    0.5000    0.5155       150
   rice_2_SD     0.6581    0.6800    0.6689       150
   rice_3_MY     0.7704    0.6933    0.7298       150
   rice_4_AP     0.5795    0.6800    0.6258       150
   rice_5_BN     0.8492    0.7133    0.7754       150
   rice_6_UN     0.7211    0.9133    0.8059       150

    accuracy                         0.8704      4500
   macro avg     0.7681    0.7390    0.7481      4500
weighted avg     0.8710    0.8704    0.8685     